In [ ]:
import pandas as pd

from wbe.inputs import get_population, get_liquid_obs_provider, get_jhu_county_data
from wbe.model import get_model, ModelSpec

pd.options.plotting.backend = "plotly"

In [ ]:
# See 04-Selection.ipynb for the selection of the Sewershed ID

FIPS_CODE = "06037"
SEWERSHED_ID = 114

population = float(get_population(FIPS_CODE))
population

In [ ]:
death_data = get_jhu_county_data("jhu_deaths_d20260226_t065541_sha8e7d05f.csv").diff().clip(lower=0)
case_data = get_jhu_county_data("jhu_confirmed_d20260226_t065534_sha8e7d05f.csv").diff().clip(lower=0)

In [ ]:
# Complete bounds of the analysis period;
# we will specify a subset for fitting to observations later

time_start, time_end = ("1 jul 2020","31 jul 2022")
times = pd.date_range(time_start, time_end)


In [ ]:
# init to weekly random process

model_info = get_model(ModelSpec(7, times, population))
epi_model = model_info.model

In [ ]:
# Plausible(ish) starting parameters

params = {
    "seed_start_time": 100.0,
    "seed_duration": 14.0,
    "seed_rate": population * 0.00003,
    "contact_rate": 0.4,
    "latent_time": 3.0,
    "recovery_time": 4.5,
    "waning_time": 160.0,
    "proc_vals": model_info.random_process_spec.default_values
}

In [ ]:
epi_model.run(params)["flows"]["infection"].to_pandas_df().plot()

In [ ]:
death_data[FIPS_CODE][time_start:time_end].rolling(7).sum().plot()

In [ ]:
case_data[FIPS_CODE][time_start:time_end].rolling(7).sum().plot()

In [ ]:
disease_state = model_info.stratification_spec.disease_state

In [ ]:
import optax
import diffrax as dfx
from jax import jit, grad, numpy as jnp
from numpyro import distributions as dist

# Diffrax settings so gradients can flow through the ODE solve
stepsize_controller = dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0)
adjoint = dfx.RecursiveCheckpointAdjoint(checkpoints=1024)
solver_kwargs = {
    "stepsize_controller": stepsize_controller,
    "adjoint": adjoint,
    "max_steps": 8000,
}

fitting_start, fitting_end = "1 nov 2020","1 may 2022"

# Observation targets: weekly JHU cases/deaths + sewershed PCR (zeros dropped)
cases_weekly = (
    case_data[FIPS_CODE][fitting_start:fitting_end]
    .reindex(times)
    .fillna(0.0)
    .rolling(7)
    .sum()[7::7]
)
deaths_weekly = (
    death_data[FIPS_CODE][fitting_start:fitting_end]
    .reindex(times)
    .fillna(0.0)
    .rolling(7)
    .sum()[7::7]
)
cases_obs = jnp.asarray(cases_weekly.to_numpy())
deaths_obs = jnp.asarray(deaths_weekly.to_numpy())

ww_series = (
    get_liquid_obs_provider()
    .subset_by("sewershed_id", SEWERSHED_ID)["pcr_conc"]
    .loc[fitting_start:fitting_end]
)
ww_series = ww_series[ww_series > 0]
ww_series = ww_series.loc[ww_series.index.intersection(times)]
ww_obs = jnp.asarray(ww_series.to_numpy())

# Observation noise scales for TruncatedNormal likelihoods (non-negative support)
case_scale = jnp.maximum(cases_obs.std(), 1.0)
death_scale = jnp.maximum(deaths_obs.std(), 1.0)
ww_scale_obs = jnp.maximum(ww_obs.std(), 1.0)

MODEL_KEYS = [
    "seed_start_time",
    "seed_duration",
    "seed_rate",
    "contact_rate",
    "latent_time",
    "recovery_time",
    "waning_time",
    "proc_vals",
]

all_params = {
    **{k: (jnp.asarray(v) if k == "proc_vals" else float(v)) for k, v in params.items()},
    # Observational scales: incidence -> cases/deaths; I -> wastewater PCR
    "case_detect": 0.3,
    "ifr": 0.01,
    "ww_scale": float(ww_series.median() / 1_000.0),
}

# Keys listed here are held fixed at their values in all_params; everything else is optimized.
FIXED_KEYS = {
    #"seed_start_time",
    "seed_duration",
    #"seed_rate",
    "latent_time",
    "recovery_time",
    #"waning_time",
    #"case_detect",
    "ifr",
    #"ww_scale",
}

fixed_params = {k: all_params[k] for k in FIXED_KEYS}
free_params = {k: v for k, v in all_params.items() if k not in FIXED_KEYS}

# Scalar free params are optimized in log-space so they stay strictly positive.
# proc_vals stay unconstrained (signed RP increments).
POSITIVE_FREE_KEYS = {k for k in free_params if k != "proc_vals"}


def constrain(free_u):
    return {k: (jnp.exp(v) if k in POSITIVE_FREE_KEYS else v) for k, v in free_u.items()}


def unconstrain(free):
    return {k: (jnp.log(v) if k in POSITIVE_FREE_KEYS else v) for k, v in free.items()}


free_u = unconstrain(free_params)
print("Fixed:", sorted(fixed_params))
print("Free (positive):", sorted(POSITIVE_FREE_KEYS))
print("Free (unconstrained):", sorted(k for k in free_params if k not in POSITIVE_FREE_KEYS))


priors = {
    "case_detect": dist.TruncatedNormal(0.3, 0.1, low=0.0, high=1.0),
}

def safe_log(x):
    return jnp.log(jnp.where(x > 0.0, x, 1e-6))

@jit
def loss(free_u):
    """Negative log-likelihood under TruncatedNormal observation models (low=0)."""
    free = constrain(free_u)
    p = {**fixed_params, **free}
    res = epi_model.run({k: p[k] for k in MODEL_KEYS}, solver_kwargs=solver_kwargs)
    ok = res["aux"].result == dfx.RESULTS.successful

    incidence_weekly = (
        res["flows"]["infection"]
        .sum(to_dims="time")
        .rolling(7, jnp.sum)
        .query(time=cases_weekly.index)
        .data
    )
    case_pred = incidence_weekly * p["case_detect"]
    death_pred = incidence_weekly * p["ifr"]
    ww_pred = res["compartments"].query(compartment=disease_state["E"]).simplify().query(time=ww_series.index).data * p["ww_scale"]
    #ww_pred = incidence_weekly * p["ww_scale"]
    safe_log_case_pred = safe_log(case_pred)
    safe_log_death_pred = safe_log(death_pred)
    safe_log_ww_pred = safe_log(ww_pred)
    #safe_log_death_pred = jnp.where(death_pred > 0.0, jnp.log(death_pred), 0.0)
    #safe_log_ww_pred = jnp.where(ww_pred > 0.0, jnp.log(ww_pred), 0.0)

    safe_log_case_obs = safe_log(cases_obs)
    safe_log_death_obs = safe_log(deaths_obs)
    safe_log_ww_obs = safe_log(ww_obs)
    #safe_log_death_obs = jnp.where(deaths_obs > 0.0, jnp.log(deaths_obs), 0.0)
    #safe_log_ww_obs = jnp.where(ww_obs > 0.0, jnp.log(ww_obs), 0.0)

    ll_priors = sum(priors[k].log_prob(free[k]) for k in priors)

    #case_ll = dist.TruncatedNormal(jnp.log(case_pred + 1e-6), 0.1, low=0.0).log_prob(jnp.log(cases_obs + 1e-6)).mean()
    #death_ll = dist.TruncatedNormal(jnp.log(death_pred + 1e-6), 0.1, low=0.0).log_prob(jnp.log(deaths_obs + 1e-6)).mean()
    #ww_ll = dist.TruncatedNormal(jnp.log(ww_pred + 1e-6), 0.1, low=0.0).log_prob(jnp.log(ww_obs + 1e-6)).mean()
    ww_ll = dist.Normal(safe_log_ww_pred, 0.1).log_prob(safe_log_ww_obs).mean()
    case_ll = dist.Normal(safe_log_case_pred, 0.1).log_prob(safe_log_case_obs).mean()
    death_ll = dist.Normal(safe_log_death_pred, 0.1).log_prob(safe_log_death_obs).mean()
    #ww_ll = 0.0
    nll = -(case_ll + death_ll + ww_ll + ll_priors)
    return jnp.where(ok, nll, 1e6)


gloss = jit(grad(loss))

start_learning_rate = 1e-3
optimizer = optax.adam(start_learning_rate)
opt_state = optimizer.init(free_u)
cur_u = free_u
best_u = cur_u
best_loss = cur_loss = loss(cur_u)
print(f"Starting loss: {best_loss:.4e}")

n_steps = 2000
for step in range(n_steps):
    grads = gloss(cur_u)
    updates, opt_state = optimizer.update(grads, opt_state)
    cur_u = optax.apply_updates(cur_u, updates)
    cur_loss = loss(cur_u)
    if cur_loss < best_loss:
        best_u = cur_u
        best_loss = cur_loss
    if step % 100 == 0 or step == n_steps - 1:
        print(f"step {step:4d}  loss={cur_loss:.4e}  best={best_loss:.4e}")

best_params = constrain(best_u)
fitted = {**fixed_params, **best_params}
print("Best scalar params:")
for k in ["contact_rate", "latent_time", "recovery_time", "waning_time", "case_detect", "ifr", "ww_scale"]:
    tag = "fixed" if k in FIXED_KEYS else "free"
    print(f"  {k} ({tag}): {float(fitted[k]):.6g}")


In [ ]:
def as_named_series(df, name):
    s = df[df.columns[0]]
    s.name = name
    return s

res = epi_model.run({k: fitted[k] for k in MODEL_KEYS}, solver_kwargs=solver_kwargs)

In [ ]:
# Wastewater obs vs modeled

ww_pred = res["compartments"].query(compartment=disease_state["E"]).simplify() * float(fitted["ww_scale"])
pd.DataFrame({
    "ww_obs": ww_series,
    "ww_pred": as_named_series(ww_pred.to_pandas_df(), "ww_pred"),
}).plot()


In [ ]:
# Cases
pd.DataFrame({
    "modeled": as_named_series((res["flows"]["infection"] * float(fitted["case_detect"])).to_pandas_df(), "cases"),
    "observed": case_data[FIPS_CODE][time_start:time_end].rolling(7).mean().rename("cases"),
}).plot()


In [ ]:
#Deaths

pd.DataFrame({
    "modeled": as_named_series((res["flows"]["infection"] * float(fitted["ifr"])).to_pandas_df(), "deaths"),
    "observed": death_data[FIPS_CODE][time_start:time_end].rolling(7).mean().rename("deaths"),
}).plot()